# Week 11 — Block 2: Guided Demo (Consuming APIs & Data Endpoints)

**DATS 6401 · Visualization of Complex Data**

Four parts, ~45 minutes.

| Part | Minutes | What it teaches |
|---|---|---|
| **A** — USGS earthquakes | ~12 | the four-step pattern, and query parameters as the interface |
| **B** — when a 200 is not a success | ~10 | why checking the status code is not enough |
| **C** — the course endpoint | ~8 | two processes, and a deliberate 404 |
| **D** — an API that needs a key | ~10 | headers, auth, tokens, latency and cost |
| **E** — into Streamlit | ~5 | caching, and failing gracefully |

Parts A, B and D need internet. Part C needs `course_endpoint.py` running in a
second terminal. Part D is an instructor demo and needs an API key; its cells
skip themselves without one.

---
## Part A — a real API, start to finish (~15 min)

The USGS earthquake catalog is public, needs no key, and takes query parameters
that actually change the question. Every API call has the same four moving
parts: a **URL**, **query parameters**, a **timeout**, and a **status check**.

In [ ]:
import requests
import pandas as pd
from datetime import date, timedelta

# Relative dates, not hardcoded ones. A notebook with starttime="2026-10-01"
# written into it quietly becomes a notebook about last year.
end = date.today()
start = end - timedelta(days=30)

r = requests.get(
    "https://earthquake.usgs.gov/fdsnws/event/1/query",   # the endpoint
    params={                                              # the query parameters
        "format": "geojson",
        "starttime": start.isoformat(),
        "endtime": end.isoformat(),
        "minmagnitude": 5,
    },
    timeout=20,                                           # never call without one
)

print("Status:", r.status_code)
print("Content-Type:", r.headers["content-type"])
r.raise_for_status()                                      # raises on 4xx/5xx

payload = r.json()                                        # JSON -> dicts and lists
print("Top-level keys:", list(payload))
print("Events returned:", payload["metadata"]["count"])

**Narrate:** *request → check status → parse JSON → DataFrame.* Everything else
this week is a variation on those four steps.

Now the parsing, which is where the geography from Weeks 9 and 10 comes back.

In [ ]:
# GeoJSON is nested: each event is a Feature with `properties` and `geometry`.
#
# The coordinates are [longitude, latitude, depth_km]. Longitude comes FIRST --
# the opposite of how we say it out loud -- and there is a third number that
# most code does not expect. Unpack the three deliberately; `lat, lon = coords`
# would silently swap the hemisphere and then crash on the extra value.
rows = []
for f in payload["features"]:
    lon, lat, depth = f["geometry"]["coordinates"]
    p = f["properties"]
    rows.append({
        # USGS gives time as epoch MILLISECONDS, not an ISO string.
        "time": pd.to_datetime(p["time"], unit="ms", utc=True),
        "mag": p["mag"],
        "place": p["place"],
        "lon": lon, "lat": lat, "depth_km": depth,
    })

quakes = pd.DataFrame(rows).sort_values("time").reset_index(drop=True)
print(quakes.shape)
quakes.head()

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Marker size is an AREA, so it goes as the square -- the same point the Altair
# `size` channel made in Week 10.
ax1.scatter(quakes.lon, quakes.lat, s=(quakes.mag - 4.5) ** 2 * 40,
            c=quakes.depth_km, cmap="viridis_r", alpha=0.75, edgecolor="none")
ax1.set(xlabel="longitude", ylabel="latitude",
        title=f"M5+ events, last 30 days (n={len(quakes)})")

ax2.scatter(quakes.mag, quakes.depth_km, alpha=0.6, edgecolor="none")
ax2.invert_yaxis()          # depth increases downward
ax2.set(xlabel="magnitude", ylabel="depth (km)", title="Magnitude against depth")

plt.tight_layout()

That left panel is a world map drawn with no projection and no basemap, which is
exactly what Week 9 warned about — plotting raw degrees on a flat grid. It is
legible enough to check the data arrived, and it is not a map you would ship.

**The query parameters are the interface.** Changing one changes the question,
and this is what a widget gets wired to later.

In [ ]:
for m in (5.0, 5.5, 6.0, 6.5):
    q = requests.get(
        "https://earthquake.usgs.gov/fdsnws/event/1/query",
        params={"format": "geojson", "starttime": start.isoformat(),
                "endtime": end.isoformat(), "minmagnitude": m},
        timeout=20,
    )
    q.raise_for_status()
    print(f"minmagnitude={m}: {q.json()['metadata']['count']:>4} events")

---
## Part B — when a 200 is not a success (~12 min)

`raise_for_status()` is necessary and not sufficient. A 200 means the server
received your request and chose to answer. It does not promise the answer is
your data.

Three live examples, all true as of today, each failing a different way.

### 1. Valid JSON, wrong shape

In [ ]:
# This exact call was in last year's materials and worked for years.
r = requests.get("https://restcountries.com/v3.1/region/americas",
                 params={"fields": "name,population,area"}, timeout=15)

print("Status:", r.status_code)                      # 200
print("Content-Type:", r.headers["content-type"])    # application/json
r.raise_for_status()                                 # passes
r.json()

In [ ]:
# Status checked, body is valid JSON. Here is last year's parsing line, unchanged:
try:
    df = pd.DataFrame([{"country": c["name"]["common"],
                        "population": c["population"]} for c in r.json()])
except TypeError as e:
    print("TypeError:", e)

# Iterating a dict yields its KEYS, so `c` is the string "success" and
# c["name"] indexes a string with a string. Notice what the error does NOT say:
# it names no API, no deprecation, no status -- it points at a line that has
# been correct for years.
print()
print("r.json() is a:", type(r.json()).__name__, "with keys", list(r.json()))

### 2. A 200 that is not JSON at all

In [ ]:
# The Census ACS API used to answer small requests without a key. It now
# returns 200 -- and an HTML page.
r2 = requests.get("https://api.census.gov/data/2023/acs/acs5",
                  params={"get": "NAME,B01001_001E", "for": "state:*"}, timeout=20)

print("Status:", r2.status_code, "| Content-Type:", r2.headers["content-type"])
try:
    r2.json()
except requests.exceptions.JSONDecodeError as e:
    print("JSONDecodeError:", e)
print()
print(r2.text[:120])

### 3. A real error whose body is not JSON either

In [ ]:
# USGS does return a proper 4xx for a bad parameter -- but the body is plain
# text, so the obvious way to read the error message fails too.
r3 = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query",
                  params={"format": "geojson", "minmagnitude": "banana"}, timeout=20)

print("Status:", r3.status_code, "| Content-Type:", r3.headers["content-type"])
try:
    r3.json()
except requests.exceptions.JSONDecodeError:
    print("r3.json() raised. Read r3.text instead:")
    print(r3.text[:170])

**Narrate:** three failures, three different layers.

| | Status | Content-Type | Parsed | What broke |
|---|---|---|---|---|
| REST Countries | 200 | `application/json` | a `dict` | the **shape** |
| Census | 200 | `text/html` | raises | the **content type** |
| USGS bad param | 400 | `text/plain` | raises | the status — and the error body is not JSON |

So the check is three checks, not one. A helper that refuses to hand back
anything we did not ask for moves every one of these failures to the line that
caused it.

In [ ]:
def get_json(url, params=None, timeout=20, expect=dict):
    """Fetch JSON, or raise where the problem actually is.

    Three checks, because the three examples above failed three different ways:
    the status, the content type, and the shape of the parsed body.
    """
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()                                      # 1. status

    ctype = r.headers.get("content-type", "")
    if "json" not in ctype:                                   # 2. content type
        raise ValueError(f"expected JSON, got {ctype!r}: {r.text[:120]}")

    payload = r.json()
    if not isinstance(payload, expect):                       # 3. shape
        raise ValueError(f"expected {expect.__name__}, got "
                         f"{type(payload).__name__}: {str(payload)[:140]}")
    return payload


checks = [
    ("USGS",           "https://earthquake.usgs.gov/fdsnws/event/1/query",
     {"format": "geojson", "starttime": start.isoformat(), "minmagnitude": 6}, dict),
    ("REST Countries", "https://restcountries.com/v3.1/region/americas",
     {"fields": "name,population"}, list),
    ("Census",         "https://api.census.gov/data/2023/acs/acs5",
     {"get": "NAME", "for": "state:*"}, list),
]

for label, url, params, expect in checks:
    try:
        out = get_json(url, params, expect=expect)
        n = out["metadata"]["count"] if isinstance(out, dict) and "metadata" in out else len(out)
        print(f"{label:16s} ok   -> {type(out).__name__}, {n}")
    except Exception as e:
        print(f"{label:16s} FAIL -> {type(e).__name__}: {str(e)[:110]}")

**The lesson to say out loud:** an API that changes under you does not announce
it. REST Countries still answers, still returns 200, still returns JSON — and
every notebook that depended on it broke silently. The three checks are what
turn that into an error message you can act on.

---
## Part C — the course endpoint: two processes (~10 min)

Everything so far called somebody else's server. Now run your own, so that
"client and server are separate processes" is something you have done rather
than been told. Next week you write what the server does; this week you only
start it.

In a **second terminal**, from this folder:

```bash
uvicorn course_endpoint:app --port 8001
```

Leave it running. It prints every request it receives — worth putting that
terminal next to this notebook so the class can watch the two halves talk.

In [ ]:
BASE = "http://127.0.0.1:8001"

# Check it is alive before anything else. A health endpoint exists so a client
# can answer "is the server up?" without asking for real data.
print(get_json(f"{BASE}/health"))
print(get_json(f"{BASE}/"))

In [ ]:
payload = get_json(f"{BASE}/data", params={"species": "setosa"})
print("Count:", payload["count"])
pd.DataFrame(payload["records"]).head(3)

In [ ]:
# Now break it on purpose: the endpoint 404s on an unknown species.
r = requests.get(f"{BASE}/data", params={"species": "dragon"}, timeout=5)
print("Status:", r.status_code)        # 404
print("Body:", r.json())               # FastAPI's errors ARE json: {"detail": ...}

try:
    r.raise_for_status()
except requests.HTTPError as e:
    print("Caught:", e)

**Narrate:** a non-200 is not an exception until *you* make it one.
`raise_for_status()` inside `try/except` is the discipline, and a client that
crashes on a 404 is a broken client.

Note the contrast with Part B: FastAPI returns its errors as JSON
(`{"detail": ...}`), so `r.json()` works on the failure here. USGS returned
plain text. You cannot assume either — which is why the helper checks.

---
## Part D — an API that needs a key (~10 min)

*Instructor demo. You do not need a key to follow this; the cells skip
themselves without one.*

Everything so far was a `GET` with the request in the URL. A model API is the
other shape, and it introduces the one piece of an HTTP request this course has
not used yet:

| | USGS and the course endpoint | a model API |
|---|---|---|
| method | `GET` | `POST` |
| the request lives in | the URL's query string | a JSON **body** |
| who you are | nobody — it is public | an **auth header** |
| what 200 guarantees | less than you think | even less |

The four steps are unchanged: request, check, parse, use.

In [ ]:
import os
import time
import requests

# The key comes from the environment, never from the notebook. A key pasted
# into a cell is a key committed to git and published with the course site.
#
#   export ANTHROPIC_API_KEY=sk-ant-...        (macOS / Linux)
#   setx ANTHROPIC_API_KEY sk-ant-...          (Windows, then reopen the shell)
HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-opus-5-5"

if not HAVE_KEY:
    print("No ANTHROPIC_API_KEY set -- the cells below will skip themselves.")
else:
    print("Key found in the environment. Calling", MODEL)

### The raw call, so the header is visible

We normally use a vendor's own library, and we will in a moment. First the
plain `requests` version, because the point of this part is the **header** — and
a library's job is to hide exactly that.

In [ ]:
PROMPT = ("In one sentence, what does the USGS earthquake API return "
          "when you ask for format=geojson?")

if HAVE_KEY:
    t0 = time.perf_counter()
    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            # This is the new idea. Not a query parameter -- a header, because
            # a URL ends up in logs, browser history and error messages.
            "x-api-key": os.environ["ANTHROPIC_API_KEY"],
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={                                  # a BODY, not params=
            "model": MODEL,
            "max_tokens": 300,
            "output_config": {"effort": "low"},  # a quick answer, not a thorough one
            "messages": [{"role": "user", "content": PROMPT}],
        },
        timeout=60,                              # models are slow; 10s is not enough
    )
    elapsed = time.perf_counter() - t0

    print("Status:", r.status_code, "|", f"{elapsed:.1f}s")
    r.raise_for_status()
    body = r.json()

    print("stop_reason:", body["stop_reason"])
    print("usage:", body["usage"])
    print()
    for block in body["content"]:                # content is a LIST of blocks,
        if block["type"] == "text":              # and not all of them are text
            print(block["text"])
else:
    print("skipped: no key")

### The same call through the SDK

This is what you would actually ship. The SDK does the retries, the timeouts,
the typed exceptions and the response parsing — but it is the same request, and
you should be able to point at each piece of it in the cell above.

```bash
pip install anthropic
```

In [ ]:
if HAVE_KEY:
    import anthropic

    client = anthropic.Anthropic()        # reads ANTHROPIC_API_KEY itself

    t0 = time.perf_counter()
    msg = client.messages.create(
        model=MODEL,
        max_tokens=300,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": PROMPT}],
    )
    sdk_elapsed = time.perf_counter() - t0

    print(f"{sdk_elapsed:.1f}s | stop_reason: {msg.stop_reason}")
    print(msg.usage)
    print()
    print(next(b.text for b in msg.content if b.type == "text"))
else:
    print("skipped: no key")

### Part B again, one level up

A model API hands you a 200 in situations that are not success at all. Two of
them matter here, and neither raises:

- **`stop_reason == "max_tokens"`** — the answer is cut off mid-sentence. HTTP
  200, valid JSON, right shape, truncated content.
- **`stop_reason == "refusal"`** — the model declined. Also a 200.

So the check grows one more line: after the status, the content type and the
shape, you check **why the model stopped**. This is the same discipline as
`get_json()`, applied to a different kind of lie.

In [ ]:
if HAVE_KEY:
    # Ask for something long and allow almost no room for it.
    cut = client.messages.create(
        model=MODEL,
        max_tokens=48,
        output_config={"effort": "low"},
        messages=[{"role": "user",
                   "content": "List every US state and its capital."}],
    )
    print("stop_reason:", cut.stop_reason)       # expect: max_tokens
    text = "".join(b.text for b in cut.content if b.type == "text")
    print("text we got:", repr(text[:120]))
    print()
    print("Nothing raised. A client that trusts this answer ships a truncated")
    print("list to its user and never knows.")
else:
    print("skipped: no key")

### What there is to visualize

The *content* of a model response is prose, which is not this course's problem.
The *call* is: it has a latency, a token count and a price, and those are
numbers that belong on a chart.

Cost comes from the token counts and the model's published rate — for
`claude-opus-5-5`, $4.00 per million input tokens and $20.00 per million output.

In [ ]:
IN_PER_MTOK, OUT_PER_MTOK = 4.00, 20.00     # claude-opus-5-5, USD per 1M tokens

if HAVE_KEY:
    import matplotlib.pyplot as plt

    # Three prompts of increasing size, so there is something to compare.
    prompts = {
        "short":  "Name one thing an API returns.",
        "medium": PROMPT,
        "long":   "Explain in three short paragraphs why a client should check "
                  "the content type of a response and not only its status code.",
    }

    runs = []
    for label, p in prompts.items():
        t0 = time.perf_counter()
        m = client.messages.create(model=MODEL, max_tokens=600,
                                   output_config={"effort": "low"},
                                   messages=[{"role": "user", "content": p}])
        runs.append({
            "prompt": label,
            "seconds": round(time.perf_counter() - t0, 2),
            "in_tok": m.usage.input_tokens,
            "out_tok": m.usage.output_tokens,
            "stop": m.stop_reason,
        })

    usage = pd.DataFrame(runs)
    usage["cost_usd"] = (usage.in_tok / 1e6 * IN_PER_MTOK
                         + usage.out_tok / 1e6 * OUT_PER_MTOK).round(5)
    display(usage)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    usage.plot.bar(x="prompt", y=["in_tok", "out_tok"], stacked=True, ax=ax1,
                   title="Tokens per call")
    ax1.set_ylabel("tokens")
    usage.plot.bar(x="prompt", y="seconds", ax=ax2, legend=False,
                   title="Latency per call", color="#d9534f")
    ax2.set_ylabel("seconds")
    plt.tight_layout()

    print(f"\nTotal for this demo: ${usage.cost_usd.sum():.5f}")
else:
    print("skipped: no key")

**Narrate:** output tokens dominate the bill — they cost five times what input
tokens do — and latency tracks output length, not prompt length. That is the
argument for `@st.cache_data` in the next part stated in dollars and seconds
rather than as advice.

This also answers a question students ask every year: *why cache at all?* A
USGS call is free and takes a quarter of a second. A model call costs money and
takes seconds. The same `@st.cache_data` decorator covers both, and the second
one is where it pays.

---
## Part E — the same calls inside Streamlit (~5 min)

Open `client_starter.py` and run it:

```bash
streamlit run client_starter.py
```

Three things to point at:

1. `@st.cache_data(ttl=300)` around the fetch — without it, every widget change
   re-requests. With it, the same parameters are answered from memory. The
   cache key is the **arguments**, which is why the fetch function takes the
   base URL and the species rather than reading them from globals.
2. The `st.error` path, and that the app stays up.
3. **Kill the server while the app is running**, then click Fetch. That
   degradation is the behavior the homework grades.

**Discussion for Block 4:** what did you cache, and what `ttl`? Country lists
change yearly; earthquake feeds change by the minute. When is caching wrong?